# Bronze Layer

## Step-1 Define schema as String

In [0]:
from pyspark.sql.types import StructType, StructField, StringType

orders_schema = StructType([
    StructField("order_id", StringType(), True),
    StructField("order_date", StringType(), True),
    StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True),
    StructField("product_name", StringType(), True),
    StructField("category", StringType(), True),
    StructField("quantity", StringType(), True),
    StructField("unit_price", StringType(), True),
    StructField("discount", StringType(), True),
    StructField("total_amount", StringType(), True),
    StructField("payment_method", StringType(), True),
    StructField("store_location", StringType(), True),
    StructField("order_status", StringType(), True)
])

customer_schema = StructType([
    StructField("customer_id", StringType(), True),
    StructField("customer_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("date_of_birth", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("registration_date", StringType(), True),
    StructField("loyalty_points", StringType(), True),
    StructField("status", StringType(), True)
])

## Step-2 Read raw as CSV

In [0]:
order_bronze_df = spark.read.format('csv').option("header", "true").schema(orders_schema).load("/Volumes/retail_project/raw_data/raw_file/retail_orders_messy.csv")

customer_bronze_df = spark.read.format('csv').option("header","true").schema(customer_schema).load("/Volumes/retail_project/raw_data/raw_file/retail_customers_messy.csv")


In [0]:
order_bronze_df.display()

customer_bronze_df.display()

## Step-3 Add Ingestion Metadata

In [0]:
from pyspark.sql.functions import current_timestamp

order_bronze_df = order_bronze_df.withColumn('ingest_time', current_timestamp())
customer_bronze_df = customer_bronze_df.withColumn('ingest_time', current_timestamp())

## Step-4 Save AS Delta Table in bronze

In [0]:
order_bronze_df.write.format("delta").mode("overwrite").saveAsTable('retail_project.bronze.bronze_order')

customer_bronze_df.write.format("delta").mode("overwrite").saveAsTable('retail_project.bronze.bronze_customer')


### Retail Voulume
### │
### ├── raw_file/
### │   ├── retail_orders/
### │   └── retail_customers/
### │
### ├── bronze/
### │   ├── bronze_orders
### │   └── bronze_customers
     
     

# Silver Layer Transformation

## STEP 1: Load Bronze Orders

In [0]:
bronze_order_df = spark.read.table("retail_project.bronze.bronze_order")

display(bronze_order_df)

## STEP 2: Fix Date Formats

In [0]:
from pyspark.sql.functions import try_to_date, col, coalesce

silver_df = bronze_order_df.withColumn(
    'order_date',
    coalesce(
        try_to_date(col('order_date'), 'yyyy-MM-dd'),
        try_to_date(col('order_date'), 'dd-MM-yyyy'),
        try_to_date(col('order_date'), 'yyyy/MM/dd'),
        try_to_date(col('order_date'), 'dd/MM/yyyy'),
    )
)

silver_df.display()

## STEP 3: Standardize Category Values

In [0]:
silver_df.select('category').distinct().display()

In [0]:
from pyspark.sql.functions import upper, trim, when

silver_df = silver_df.withColumn(
    "category",
    upper(trim(col("category")))
)

silver_df = silver_df.withColumn(
        'category',when(col('category')=='ELECTRONIC','ELECTRONICS').
        when(col('category')=='HOME AND LIVING','HOME & LIVING').
        when(col('category')=='FASHON','FASHION').
        otherwise(col('category')))

## STEP 4: Remove Duplicate Orders

In [0]:
from pyspark.sql.functions import count, countDistinct

silver_df.select(count('order_id'),countDistinct('order_id')).display()

In [0]:
silver_df = silver_df.dropDuplicates(['order_id'])

silver_df.select(count('order_id'),countDistinct('order_id')).display()

## STEP 5: Handle Null Values discount

In [0]:
from pyspark.sql.functions import when

silver_df = silver_df.withColumn('discount',
                                 when(col('discount').isNull(),'0').when(col('discount') == 'ten','0.1').otherwise(col('discount')))

In [0]:
silver_df.display()

## STEP 6: Convert Discount from "5%" to Decimal

In [0]:
from pyspark.sql.functions import regexp_replace

silver_df = silver_df.withColumn("discount",
                                 when(col("discount").like("5%"),
                                     (regexp_replace("discount","5%","0.05"))
                                     ).when(col("discount").like("10%"),
                                            (regexp_replace("discount","10%","0.1"))
                                     ).otherwise(col("discount")))

silver_df.display()

## STEP 7: Remove Negative Quantity

In [0]:
silver_df = silver_df.filter(col("quantity") >=0)

In [0]:
silver_df.display()

## STEP 8: Cast unit_price to Decimal

In [0]:
from pyspark.sql.types import DecimalType

silver_df = silver_df.withColumn("unit_price",
                                 col("unit_price").try_cast(DecimalType(10,2)))

silver_df.display()

## STEP 9: Recalculate Total Amount

In [0]:
silver_df = silver_df.withColumn("total_amount",
                                 col("quantity").cast("int") * col("unit_price") * (1 - col("discount").cast("double"))
                                 )

In [0]:
silver_df.display()

In [0]:
silver_df.write.format("delta").mode("overwrite").saveAsTable("retail_project.silver.silver_order")

## Retail Customers – Silver Layer Cleaning | Databricks End-to-End Project

### STEP 1: Read Bronze Table

In [0]:
customer_bronze_df = spark.table("retail_project.bronze.bronze_customer")

display(customer_bronze_df)

### STEP 2: Trim Spaces

In [0]:
from pyspark.sql.functions import trim, col

customer_df = customer_bronze_df.select(
    trim(col("customer_id")).alias("customer_id"),
    trim(col("customer_name")).alias("customer_name"),
    trim(col("email")).alias("email"),
    trim(col("phone")).alias("phone"),
    trim(col("gender")).alias("gender"),
    trim(col("date_of_birth")).alias("date_of_birth"),
    trim(col("city")).alias("city"),
    trim(col("state")).alias("state"),
    trim(col("registration_date")).alias("registration_date"),
    trim(col("loyalty_points")).alias("loyalty_points"),
    trim(col("status")).alias("status"),
    col("ingest_time")
)

display(customer_df)

### STEP 3: Remove Duplicate Customers

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number,desc

window_spec = Window.partitionBy("customer_id").orderBy(desc("ingest_time"))

customer_df = customer_df.withColumn("row_num",
                                      row_number().over(window_spec))

customer_df = customer_df.filter("row_num = 1").drop("row_num")

customer_df.display()

### STEP 4: Fix and Standardize Status

In [0]:
from pyspark.sql.functions import upper, when

customer_df = customer_df.withColumn("status",
                                     upper(col("status"))
                                     )

customer_df = customer_df.withColumn("status",
                                     when(col("status").isin("ACTIVE","INACTIVE"), col("status"))
                                          .otherwise("UNKNOWN") 
                                          )

customer_df.display()

### STEP 5: Convert loyalty_points to Integer

In [0]:
from pyspark.sql.functions import regexp_replace

customer_df = customer_df.withColumn("loyalty_points",
                                      regexp_replace(col("loyalty_points"),"[^0-9]","")
                                      )
 
customer_df = customer_df.withColumn("loyalty_points",
                                      col("loyalty_points").try_cast("int")
                                      )

In [0]:
customer_df.display()

### STEP 6: Validate Email Format

In [0]:
customer_df = customer_df.withColumn("email_valid",
                                    col("email").rlike("^[A-Za-z0-9+_.-]+@[A-Za-z0-9.-]+\\.[A-Za-z]{2,}$")
                                    )

customer_df = customer_df.filter(col("email_valid") == True)
customer_df.display()

### STEP 7: Fix Date Formats

In [0]:
from pyspark.sql.functions import to_date, coalesce

customer_df = customer_df.withColumn("dob_parsed",
                                     coalesce(
                                         try_to_date(col('date_of_birth'),'yyyy-MM-dd'),
                                         try_to_date(col('date_of_birth'),'dd-MM-yyyy'),
                                         try_to_date(col('date_of_birth'),'yyyy/MM/dd'),
                                         try_to_date(col('date_of_birth'),'dd/MM/yyyy')
                                        )
                                    )

customer_df = customer_df.withColumn("registration_parsed",
                                     coalesce(
                                        try_to_date(col('registration_date'),'yyyy-MM-dd'),
                                        try_to_date(col('registration_date'),'dd-MM-yyyy'),
                                        try_to_date(col('registration_date'),'yyyy/MM/dd'),
                                        try_to_date(col('registration_date'),'dd/MM/yyyy')
                                        )
                                    )

customer_df.display()

In [0]:
customer_df = customer_df.drop("date_of_birth","registration_date").withColumnRenamed("dob_parsed","date_of_birth").withColumnRenamed("registration_parsed","registration_date")

In [0]:
customer_df.display()

### STEP 8: Standardize Gender

In [0]:
customer_df = customer_df.withColumn("gender",
                                     upper(col("gender"))
                                     )

customer_df = customer_df.withColumn("gender",
                                     when(col("gender") == "MALE", "M").when(col("gender") == "FEMALE", "F").otherwise("U")
                                     )

In [0]:
customer_df.display()

In [0]:
customer_df.write.format("delta").mode("overwrite").saveAsTable("retail_project.silver.silver_customer")

# Gold Layer in Databricks – Building Fact & Dimension Tables | Retail End-to-End Project

### STEP 1: Load Silver Tables

In [0]:
customer_df = spark.read.table("retail_project.silver.silver_customer")

order_df = spark.read.table("retail_project.silver.silver_order")

customer_df.display()
order_df.display()

### STEP 2: Create Customer Dimension

In [0]:
dim_customer = customer_df.select("customer_id",
                                  "customer_name",
                                  "email",
                                  "city",
                                  "state",
                                  "loyalty_points",
                                  "status")

dim_customer.display()

In [0]:
dim_customer.write.format("delta").mode("overwrite").saveAsTable("retail_project.gold.gold_dim_customer")

### STEP 3: Create Fact Sales Table

In [0]:
fact_sales = order_df.join(customer_df,
                            order_df.customer_id == customer_df.customer_id,
                            "left"
                            ).select(order_df.order_id,
                                     order_df.order_date,
                                     order_df.customer_id,
                                     customer_df.customer_name,
                                     order_df.product_id,
                                     order_df.product_name,
                                     order_df.category,
                                     order_df.quantity,
                                     order_df.unit_price,
                                     order_df.discount,
                                     order_df.total_amount,
                                     customer_df.city,
                                     customer_df.state,
                                     customer_df.gender,
                                     customer_df.loyalty_points
                                     )
                            
from pyspark.sql.functions import year, month, dayofmonth

order_enriched = fact_sales.withColumn("Year",year("order_date")).withColumn("Month",month("order_date")).withColumn("Day",dayofmonth("order_date"))

order_enriched.display()

In [0]:
from pyspark.sql.functions import sum, avg, countDistinct, min, max

gold_df = order_enriched.groupBy(
    "product_id",
    "product_name",
    "customer_id",
    "customer_name",
    "category",
    "Year",
    "Month",
    "order_date",
    "gender",
    "city",
    "state",
    "loyalty_points"
).agg(
    sum("total_amount").alias("Total_revenue"),
    sum("quantity").alias("Total_quantity"),
    countDistinct("order_id").alias("Total_orders"),
    avg("total_amount").alias("Avg_revenue"),
    avg("quantity").alias("Avg_quantity"),
    avg("loyalty_points").alias("Avg_loyalty_points"),
    min("unit_price").alias("Min_price"),
    max("unit_price").alias("Max_price")
)

gold_df.display()

In [0]:
gold_df.write.format("delta").mode("overwrite").saveAsTable("retail_project.gold.gold_fact_sales")